# UABR-SLM · Notebook 03b — Option A: authentic fact-sentence re-injection (continued pretraining)

**G1 fallback** (PLAN §11 G1 fail → "re-inject authentic Wikipedia text"; team decisions 10 Nov 2026, PLAN §16).
GPU **T4 ×2** · ~40–60 min · output ~0.6 GB.

**Why:** notebook 04 v1 showed no stored knowledge — the randomized +0/+1/+4/+16 *article-copy* exposure had
**no effect** on the gold answer's log-probability (High − Control −0.15 nats, CI [−0.57, +0.29]).

**What this notebook does (team decisions):**
1. **Fact sentences** = authentic sentences from the notebook-01 corpus that name the **subject** and contain a
   **gold answer alias**, taken from the subject's article, the object's article and the articles of the question's
   top-20 BM25 windows. *(String-match proxy for "states the fact" — a 100-sentence audit sample is exported.)*
2. **4,000 subjects** with ≥ 1 fact sentence (in-main questions, **all splits**), stratified by relation × popularity,
   **1,000 each at doses 0 / 16 / 64 / 256** injected sentences **per fact**, cycling through **all** of the fact's
   sentences (wording variety where it exists).
3. **Continued pretraining from `model_final.pt`**: injected sentences are **~25 %** of tokens, **~75 %** replayed
   random 512-token blocks of the original pretraining stream (against forgetting); one shuffled pass.
4. The old article-copy exposure groups are **replaced** by these doses for all analyses.

**Training recipe (my defaults — change in the config cell):** fresh AdamW, peak lr **2e-4**, 50 warm-up steps,
cosine to 2e-5, 131k tokens/step, fp16. **Forgetting check:** validation loss must not rise by more than 0.05 nats.

**Inputs (Add Input → Your Work):** outputs of notebooks **00, 01, 02, 03**.
**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** · **Save & Run All**.

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.data import tokenizer as T
from uabr.data import fact_sentences as FS
from uabr.data.wiki_corpus import normalize_title, TokenWriter
from uabr.eval.generate import load_model
from uabr.eval.probes import answer_span, score_continuations

## 2 · Configuration and inputs

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq
OUT = "/kaggle/working/03b_reinject" if ON_KAGGLE else os.path.abspath("outputs/03b_reinject")
os.makedirs(OUT, exist_ok=True)
if ON_KAGGLE: assert torch.cuda.is_available(), "No GPU - set Accelerator to GPU T4 x2"

rcfg = FS.ReinjectConfig(n_subjects=int(os.environ.get("N_SUBJECTS", 4000)), fact_share=0.25)
LR, MIN_LR, WARMUP = 2e-4, 2e-5, int(os.environ.get("WARMUP_STEPS", 50))
MAX_FORGET_NATS = 0.05

POPQA = find_input("00_popqa_splits/popqa_prepared.jsonl", "POPQA_PREPARED")
ART = find_input("01_corpus/corpus/articles.parquet", "ARTICLES")
TRAIN_BIN = find_input("01_corpus/pretrain/train.bin", "TRAIN_BIN")
VAL_BIN = find_input("01_corpus/pretrain/val.bin", "VAL_BIN")
TOK = find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER")
TOP20 = find_input("02_bm25/retrieval/popqa_top20.parquet", "TOP20")
WINS = find_input("02_bm25/retrieval/windows_retrieved.parquet", "WINDOWS")
BASE = find_input("03_pretrain/model_final.pt", "MODEL")
tok = T.load(TOK)
popqa = pd.read_json(POPQA, lines=True)
main = popqa[popqa["in_main"]].reset_index(drop=True)
print(f"in-main questions {len(main):,} | base model {BASE}")
rcfg.to_dict()

## 3 · Gather source articles (subject, object, top-20 window articles)

In [ ]:
titles = pq.read_table(ART, columns=["title"]).column("title").to_pylist()
title_row = {}
for i, t in enumerate(titles):
    title_row.setdefault(normalize_title(t), i)
top = pd.read_parquet(TOP20, columns=["id", "doc_ids"])
wins = pd.read_parquet(WINS, columns=["doc_id", "article_idx"])
doc_art = dict(zip(wins["doc_id"], wins["article_idx"]))
top_docs = dict(zip(top["id"], top["doc_ids"]))

sources = {}
for r in main.itertuples(index=False):
    rows = [title_row.get(normalize_title(r.s_wiki_title)), title_row.get(normalize_title(r.o_wiki_title))]
    rows += [doc_art[d] for d in top_docs.get(r.id, []) if d in doc_art]
    sources[r.id] = list(dict.fromkeys(x for x in rows if x is not None))
need = sorted({x for v in sources.values() for x in v})
text_col = pq.read_table(ART, columns=["text"]).column("text")
TEXT = dict(zip(need, text_col.take(need).to_pylist())); del text_col
LOWER = {k: v.lower() for k, v in TEXT.items()}
stage(f"{len(need):,} source articles loaded ({sum(map(len, TEXT.values()))/1e6:.0f}M chars)")

## 4 · Extract authentic fact sentences

In [ ]:
facts = []
for i, r in enumerate(main.itertuples(index=False)):
    names = [r.subj] + list(r.s_aliases)
    nl = [n.lower() for n in names if len(n) >= rcfg.subj_min_chars]
    gl = [g.lower() for g in r.gold_aliases if len(g) >= rcfg.gold_min_chars]
    cand = [TEXT[a] for a in sources[r.id] if any(n in LOWER[a] for n in nl) and any(g in LOWER[a] for g in gl)]
    sents = FS.fact_sentences_for(cand, names, r.gold_aliases, rcfg) if cand else []
    facts.append({"id": r.id, "subj_id": r.subj_id, "subj_key": normalize_title(r.s_wiki_title) or str(r.subj_id),
                  "prop": r.prop, "pop_bin": r.pop_bin, "split": r.split, "n_sentences": len(sents), "sentences": sents})
    if (i + 1) % 2000 == 0: stage(f"  {i+1:,}/{len(main):,}")
facts = pd.DataFrame(facts)
elig = facts[facts["n_sentences"] > 0]
print(f"in-main questions with >=1 fact sentence: {len(elig):,} / {len(facts):,} ({len(elig)/len(facts):.0%})")
display(facts.groupby("prop")["n_sentences"].agg(eligible=lambda s: (s > 0).mean(), median="median").round(2))
print("sentences per eligible fact:", elig["n_sentences"].describe()[["mean", "50%", "max"]].round(1).to_dict())

## 5 · Assign doses (4,000 subjects, 1,000 per dose)

In [ ]:
subj = (elig.groupby("subj_key").agg(primary_prop=("prop", lambda s: s.value_counts().index[0]), pop_bin=("pop_bin", "first"))
            .reset_index().rename(columns={"subj_key": "title_n"}))
doses = FS.assign_doses(subj, rcfg).rename(columns={"title_n": "subj_key"})
facts = facts.merge(doses[["subj_key", "dose_group", "dose"]], on="subj_key", how="left")
facts["dose_group"] = facts["dose_group"].fillna("undosed"); facts["dose"] = facts["dose"].fillna(0).astype(int)
facts.loc[facts["n_sentences"] == 0, ["dose_group", "dose"]] = ["no_sentence", 0]
print(f"eligible subjects {len(subj):,} -> selected {len(doses):,}")
display(pd.crosstab(facts["dose_group"], facts["split"], margins=True))
display(pd.crosstab(doses["primary_prop"], doses["dose_group"]))

## 6 · Build the continued-pretraining stream (25 % fact sentences, 75 % replay)

In [ ]:
docs = FS.injection_docs(facts[facts["dose"] > 0])
fact_arrays = []
for s in range(0, len(docs), 4096):
    fact_arrays += [T.wrap_doc(a) for a in T.encode_batch(tok, docs[s:s+4096])]
FACT_TOKENS = int(sum(len(a) for a in fact_arrays))
REPLAY_TOKENS = int(FACT_TOKENS * (1 - rcfg.fact_share) / rcfg.fact_share)
train_mm = np.memmap(TRAIN_BIN, dtype=np.uint16, mode="r")
n_blocks = REPLAY_TOKENS // 512
rng = np.random.default_rng(rcfg.seed)
starts = rng.integers(0, len(train_mm) - 513, size=n_blocks)
order = [("f", i) for i in range(len(fact_arrays))] + [("r", i) for i in range(n_blocks)]
perm = rng.permutation(len(order))
w = TokenWriter(f"{OUT}/reinject.bin")
for k in perm:
    kind, i = order[k]
    if kind == "f": w.write(fact_arrays[i], "fact", "")
    else: w.write(np.asarray(train_mm[starts[i]:starts[i] + 512]), "replay", str(int(starts[i])))
idx = w.close(); idx.to_parquet(f"{OUT}/reinject_index.parquet", index=False)
stage(f"fact docs {len(docs):,} = {FACT_TOKENS/1e6:.1f}M tokens | replay {REPLAY_TOKENS/1e6:.1f}M | stream {w.n/1e6:.1f}M tokens "
      f"(fact share {FACT_TOKENS/w.n:.1%})")
facts.to_parquet(f"{OUT}/facts_doses.parquet", index=False)
aud = facts[facts["n_sentences"] > 0].sample(min(100, len(elig)), random_state=1)
pd.DataFrame({"id": aud["id"], "prop": aud["prop"], "sentence": aud["sentences"].map(lambda s: s[0]),
              "states_fact_manual": ""}).merge(main[["id", "subj", "gold_aliases"]], on="id").to_csv(f"{OUT}/fact_sentence_audit.csv", index=False)

## 7 · Continued pretraining from `model_final.pt`

In [ ]:
base_state = torch.load(BASE, map_location="cpu", weights_only=False)
RUN = dict(train_bin=f"{OUT}/reinject.bin", val_bin=VAL_BIN, out_dir=OUT, model=base_state["model_cfg"],
           seq_len=512, global_batch_seqs=int(os.environ.get("GLOBAL_BATCH_SEQS", 256)),
           micro_batch_seqs=int(os.environ.get("MICRO_BATCH_SEQS", 32)), lr=LR, min_lr=MIN_LR, warmup_steps=WARMUP,
           weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0, precision="auto", compile=False, seed=1234,
           data_seed=8765, eval_every=int(os.environ.get("EVAL_EVERY", 100)), eval_tokens=int(os.environ.get("EVAL_TOKENS", 2_000_000)),
           log_every=10, ckpt_minutes=30, max_hours=11.0, max_steps=0, init_from=BASE)
RUN["model"].pop("max_seq_len", None)
json.dump(RUN, open(f"{OUT}/run_config.json", "w"), indent=2); del base_state
NPROC = max(1, torch.cuda.device_count())
env = {**os.environ, "PYTHONPATH": SRC + os.pathsep + os.environ.get("PYTHONPATH", ""),
       "NCCL_P2P_DISABLE": os.environ.get("NCCL_P2P_DISABLE", "1"), "OMP_NUM_THREADS": "2"}
cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NPROC}",
       "-m", "uabr.train.pretrain", "--config", f"{OUT}/run_config.json"]
with open(f"{OUT}/train_stdout.log", "a") as logf:
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: print(line, end=""); logf.write(line)
    assert p.wait() == 0, "continued pretraining failed - see train_stdout.log"
SUMMARY = json.load(open(f"{OUT}/summary.json"))
log = pd.read_csv(f"{OUT}/train_log.csv"); va = log.dropna(subset=["val_loss"])
V0, V1 = float(va["val_loss"].iloc[0]), float(va["val_loss"].iloc[-1])
print(f"val loss: base {V0:.4f} -> after re-injection {V1:.4f} ({V1 - V0:+.4f} nats)")

## 8 · Sanity check: did injection take? (answer log-prob inside its own fact sentence)

In [ ]:
DEV = "cuda" if torch.cuda.is_available() else "cpu"
gold = dict(zip(main["id"], main["gold_aliases"]))
chk = []
for g, sub in facts[facts["dose_group"].isin(list(rcfg.doses))].groupby("dose_group"):
    for r in sub.sample(min(150, len(sub)), random_state=3).itertuples(index=False):
        sp = answer_span(r.sentences[0], gold[r.id])
        if sp: chk.append((g, r.id, r.sentences[0][:sp[0]].rstrip(), sp[1]))
res = []
for name, path in [("base", BASE), ("reinjected", f"{OUT}/model_final.pt")]:
    m = load_model(path, DEV)
    for g, qid, prefix, ans in chk:
        p_ids = [T.BOS_ID] + tok.encode(prefix, add_special_tokens=False).ids
        a_ids = tok.encode(" " + ans, add_special_tokens=False).ids
        _, mean = score_continuations(m, p_ids, [a_ids], DEV)
        res.append({"model": name, "dose_group": g, "id": qid, "lp": float(mean[0])})
    del m
res = pd.DataFrame(res)
tab = res.pivot_table(index="dose_group", columns="model", values="lp", aggfunc="mean").reindex(list(rcfg.doses))
tab["gain"] = tab["reinjected"] - tab["base"]
display(tab.round(3))
D256 = float(tab.loc["d256", "gain"] - tab.loc["d0", "gain"])
print(f"verbatim answer log-prob gain, d256 vs d0: {D256:+.3f} nats")

## 9 · Gate 03b

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("training_completed", SUMMARY["completed"], f"{SUMMARY['steps_done']}/{SUMMARY['total_steps']} steps")
add("no_forgetting_le_0.05_nats", V1 - V0 <= MAX_FORGET_NATS, f"val loss {V0:.4f} -> {V1:.4f}")

dsub = doses["dose_group"].value_counts(normalize=True)
add("doses_balanced", (dsub - 0.25).abs().max() <= 0.01, dsub.round(3).to_dict())
add("injection_took_d256_gt_d0", D256 > 0, f"{D256:+.3f} nats (verbatim answer log-prob)")
add("fact_share_about_25pct", abs(FACT_TOKENS / w.n - rcfg.fact_share) < 0.02, f"{FACT_TOKENS / w.n:.1%}", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "03b_reinject", "status": status, "checks": checks, "val_loss": [V0, V1], "verbatim_gain_d256_vs_d0": D256,
           "n_subjects_dosed": len(doses), "fact_tokens": FACT_TOKENS, "stream_tokens": int(w.n),
           "config": rcfg.to_dict(), "train": RUN, "code_commit": CODE_COMMIT},
          open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print("\nGATE 03b:", status)
assert not fails, "Re-injection gate failed."

## Next
Run **notebook 04 v2** with outputs **00, 01, 02, 03, 03b** attached. It probes both models (base and re-injected)
with zero-shot cloze, the ranking probe and the verbatim probe, and evaluates G1 using the new doses.
Label `fact_sentence_audit.csv` (100 rows) to report the precision of the fact-sentence filter.